# 1. Dataset Preprocessing

Load images from `datasetScrap3_augmented/`, resize to 224×224, normalize, split into train/val/test sets (70/15/15).

**Dataset**: datasetScrap3_augmented (1000 images: 500 jernih + 500 keruh)

In [1]:
# ============================================================================
# 1_Preprocessing.ipynb - Load and Preprocess River Turbidity Dataset
# ============================================================================
# Purpose: Load 1000 images dari datasetScrap3_augmented/, preprocess (resize + normalize),
#          dan split menjadi train/val/test sets (70/15/15)
# Output: data_processed/ folder dengan numpy arrays untuk training
# UPDATED: Now using AUGMENTED dataset (datasetScrap3_augmented: 500 jernih + 500 keruh)
# ============================================================================

# Import required libraries
import os
import numpy as np              # Numerical computing
import cv2                      # Computer vision (image processing)
from pathlib import Path        # File path handling (cross-platform)
from sklearn.model_selection import train_test_split  # Data splitting
import json                     # Save metadata

print("Libraries imported successfully")

Libraries imported successfully


## 1. Load Images

In [3]:
# Step 1: Define paths and load images
# ============================================================================
# dataset_path: Path ke folder berisi image (subfolder: jernih/, keruh/)
# output_path: Akan menyimpan preprocessed data
# UPDATED: Gunakan datasetScrap3_augmented/ (1000 images: 500 jernih + 500 keruh)
dataset_path = Path('datasetScrap3_augmented')  # ← UPDATED: Using augmented dataset!
output_path = Path('data_processed')   # ← Save preprocessed data

# Gunakan glob pattern **/*.jpg untuk recursive search (mencari di subfolders)
# Ini penting karena images di datasetScrap3_augmented/jernih/ dan datasetScrap3_augmented/keruh/
image_files = list(dataset_path.glob('**/*.jpg')) + list(dataset_path.glob('**/*.png'))
print(f"Total images found: {len(image_files)}")

# Step 2: Categorize images by label berdasarkan folder name
# ============================================================================
# Label: 0 = Clear (Jernih), 1 = Turbid (Keruh)
# Strategy: Detect dari parent folder name (jernih, keruh) atau filename
labels = {}
image_paths = []

for img_file in sorted(image_files):
    # Ambil nama folder parent (tempat file berada)
    parent_name = img_file.parent.name.lower()
    
    # Cek apakah parent folder bernama 'jernih' atau 'clear'
    if 'jernih' in parent_name or 'clear' in parent_name:
        label = 0  # Clear water
    # Cek apakah parent folder bernama 'keruh' atau 'turbid'
    elif 'keruh' in parent_name or 'turbid' in parent_name:
        label = 1  # Turbid water
    else:
        # Jika folder name tidak jelas, cek filename
        if 'keruh' in img_file.name.lower():
            label = 1
        elif 'jernih' in img_file.name.lower():
            label = 0
        else:
            continue  # Skip jika tidak bisa tentukan label
    
    # Simpan path dan label
    image_paths.append((str(img_file), label))
    labels[str(img_file)] = label

print(f"Images with labels: {len(image_paths)}")
print(f"Label distribution: Clear={sum(1 for _, l in image_paths if l==0)}, Turbid={sum(1 for _, l in image_paths if l==1)}")

Total images found: 1000
Images with labels: 1000
Label distribution: Clear=500, Turbid=500


## 2. Image Preprocessing Function

In [4]:
# Step 2: Define image preprocessing function
# ============================================================================
# Fungsi ini akan digunakan untuk standardisasi semua images sebelum training
def preprocess_image(img_path, target_size=(224, 224)):
    """
    Load dan preprocess single image untuk ResNet18 model
    
    Args:
        img_path (str): Path ke image file
        target_size (tuple): Target size untuk resize (default 224x224 untuk ResNet18)
    
    Returns:
        np.array: Preprocessed image atau None jika gagal load
    
    Steps:
        1. Load image menggunakan OpenCV (BGR format)
        2. Convert BGR → RGB (OpenCV default adalah BGR, PyTorch expect RGB)
        3. Resize ke 224x224 (standar input untuk ResNet18)
        4. Normalize ke range [0, 1] (float32 / 255.0)
    """
    # Load image dari file (OpenCV membaca dalam BGR format)
    img = cv2.imread(img_path)
    if img is None:
        return None  # Return None jika file tidak valid
    
    # Convert BGR ke RGB (penting untuk color consistency dengan ImageNet)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resize ke 224x224 menggunakan linear interpolation
    # (ResNet18 expect input 224x224)
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
    
    # Normalize pixel values ke [0, 1] range
    # Ini important untuk neural networks (converge lebih cepat)
    img = img.astype('float32') / 255.0
    
    return img

print("Preprocessing function defined")

Preprocessing function defined


## 3. Load and Process All Images

In [5]:
# Step 3: Load semua images dengan progress tracking
# ============================================================================
# Initialize lists untuk menyimpan images dan labels yang valid
images = []           # List untuk store preprocessed images
valid_labels = []     # List untuk store corresponding labels
valid_paths = []      # List untuk store file paths (useful untuk debugging)
failed = 0            # Counter untuk images yang gagal di-load

print(f"Loading {len(image_paths)} images...")

# Loop melalui semua image files
for idx, (img_path, label) in enumerate(image_paths):
    # Show progress setiap 100 images
    if (idx + 1) % 100 == 0:
        print(f"  {idx + 1}/{len(image_paths)} loaded...")
    
    # Preprocess image
    img = preprocess_image(img_path)
    
    # Jika berhasil preprocess, tambahkan ke lists
    if img is not None:
        images.append(img)
        valid_labels.append(label)
        valid_paths.append(img_path)
    else:
        failed += 1  # Count failed images

# Summary setelah semua images loaded
print(f"\n✅ Loaded: {len(images)} images")
print(f"❌ Failed: {failed} images")

# Convert lists ke numpy arrays (efficient untuk matrix operations)
X = np.array(images)   # Shape: (n_images, 224, 224, 3) - RGB images
y = np.array(valid_labels)  # Shape: (n_images,) - Labels (0 atau 1)

# Print dataset statistics
print(f"\nDataset shape: {X.shape}")  # Should be (1000, 224, 224, 3) untuk semua 1000 images
print(f"Labels shape: {y.shape}")
print(f"Data type: {X.dtype}")  # float32 (dari normalisasi)
print(f"Value range: [{X.min():.3f}, {X.max():.3f}]")  # Should be [0.0, 1.0]

Loading 1000 images...
  100/1000 loaded...
  100/1000 loaded...
  200/1000 loaded...
  200/1000 loaded...
  300/1000 loaded...
  300/1000 loaded...
  400/1000 loaded...
  400/1000 loaded...
  500/1000 loaded...
  500/1000 loaded...
  600/1000 loaded...
  600/1000 loaded...
  700/1000 loaded...
  700/1000 loaded...
  800/1000 loaded...
  800/1000 loaded...
  900/1000 loaded...
  900/1000 loaded...
  1000/1000 loaded...

✅ Loaded: 1000 images
❌ Failed: 0 images

Dataset shape: (1000, 224, 224, 3)
Labels shape: (1000,)
Data type: float32
  1000/1000 loaded...

✅ Loaded: 1000 images
❌ Failed: 0 images

Dataset shape: (1000, 224, 224, 3)
Labels shape: (1000,)
Data type: float32
Value range: [0.000, 1.000]
Value range: [0.000, 1.000]


## 4. Split Dataset (Train/Val/Test = 70/15/15)

In [6]:
# Step 4: Split dataset menjadi Train/Val/Test (70/15/15)
# ============================================================================
# WHY stratify=y? 
#   - Stratified split memastikan label distribution tetap seimbang di setiap split
#   - Important ketika dataset imbalanced (contoh: 95 jernih, 109 keruh)
# 
# random_state=42: Set seed untuk reproducibility (hasil akan sama setiap run)

# Split 1: 70% training, 30% temporary (untuk di-split lagi menjadi val & test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.30,      # 30% untuk val+test
    random_state=42,     # Reproducible split
    stratify=y           # Balance labels di setiap split
)

# Split 2: Split temporary 50/50 menjadi validation (15%) dan test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.50,      # 50% dari temp = 15% dari total
    random_state=42,
    stratify=y_temp
)

# Print split information
print(f"Train set: {X_train.shape} (labels: {np.bincount(y_train)})")
print(f"Val set: {X_val.shape} (labels: {np.bincount(y_val)})")
print(f"Test set: {X_test.shape} (labels: {np.bincount(y_test)})")
print(f"\nTotal: {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} images")

Train set: (700, 224, 224, 3) (labels: [350 350])
Val set: (150, 224, 224, 3) (labels: [75 75])
Test set: (150, 224, 224, 3) (labels: [75 75])

Total: 1000 images


## 5. Save Processed Data

In [7]:
# Step 5: Save preprocessed data ke disk
# ============================================================================
# Mengapa save ke .npy files?
#   - Faster loading untuk training (dibanding load dari image files setiap epoch)
#   - Smaller file size vs raw images
#   - Easy to load dengan numpy: X_train = np.load('train/images.npy')

# Create directory structure
output_path.mkdir(exist_ok=True)  # Create data_processed/ jika belum ada
for split in ['train', 'val', 'test']:
    (output_path / split).mkdir(exist_ok=True)  # Create train/, val/, test/ subdirs

# Save training set
# images.npy: (143, 224, 224, 3) - all training images
# labels.npy: (143,) - corresponding labels
np.save(output_path / 'train' / 'images.npy', X_train)
np.save(output_path / 'train' / 'labels.npy', y_train)

# Save validation set
# images.npy: (31, 224, 224, 3) - all validation images
# labels.npy: (31,) - corresponding labels
np.save(output_path / 'val' / 'images.npy', X_val)
np.save(output_path / 'val' / 'labels.npy', y_val)

# Save test set
# images.npy: (30, 224, 224, 3) - all test images (untuk final evaluation)
# labels.npy: (30,) - corresponding labels
np.save(output_path / 'test' / 'images.npy', X_test)
np.save(output_path / 'test' / 'labels.npy', y_test)

# Save metadata (untuk reference dan documentation)
metadata = {
    'total_images': len(X),
    'train_size': X_train.shape[0],
    'val_size': X_val.shape[0],
    'test_size': X_test.shape[0],
    'image_size': (224, 224),        # ResNet18 standard input size
    'num_classes': 2,                # Clear & Turbid
    'class_names': ['Clear (Jernih)', 'Turbid (Keruh)'],
    'preprocessing': 'Resized to 224x224, normalized to [0, 1]'
}

with open(output_path / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Print summary
print("✅ Preprocessing complete!")
print(f"\nSaved to: {output_path}")
print(f"\nDataset summary:")
print(json.dumps(metadata, indent=2))

✅ Preprocessing complete!

Saved to: data_processed

Dataset summary:
{
  "total_images": 1000,
  "train_size": 700,
  "val_size": 150,
  "test_size": 150,
  "image_size": [
    224,
    224
  ],
  "num_classes": 2,
  "class_names": [
    "Clear (Jernih)",
    "Turbid (Keruh)"
  ],
  "preprocessing": "Resized to 224x224, normalized to [0, 1]"
}
